In [ ]:
sentence: str = r"""What began in 1988 as a standard for character encoding has grown into a powerful portfolio of open source standards, code, tools, libraries, and products that ensure global language support, interoperability, and resiliency across billions of devices.

Behind what most take for granted on screens today is the Unicode Consortium, the non- profit open source, open standards body for the internationalization of software and services. Unicode is embedded in every major operating system and used on more than 20 billion devices worldwide. It may be the most widely deployed technology ever.

🔧 Unicode Is Essential to the Global Tech Stack.
Today’s digital economy spans the globe. A traveler books a flight in Swahili. A payment system processes addresses in Hindi. Commerce engines, databases, and websites support names, currencies, and text in hundreds of languages—from English to Japanese —thanks to Unicode. It provides the foundation for internationalization and the architecture for localization.

🚀 Unicode Accelerates Time to Market.
Building software and services with Unicode by design powers global-ready software and services. Organizations using Unicode libraries and architecture reduce development time and scale faster. What once took months or years to support new languages and locales now takes days. Unicode ensures users around the world can access digital content in their native scripts in ways they expect
"""

### TikTokenizer

In [2]:
import tiktoken
from tiktoken_ext.openai_public import (
    ENDOFPROMPT,
    FIM_MIDDLE,
    FIM_PREFIX,
    FIM_SUFFIX,
)

print(f"{tiktoken.list_encoding_names()=}")

base_tok = tiktoken.get_encoding("gpt2")
print(f"{base_tok._pat_str=}")
print(f"{base_tok.n_vocab=}")
print(
    f"mergable_rank={[{i: k} for i, k in base_tok._mergeable_ranks.items()][1010:1020]}"
)
print(f"special_tok={base_tok.special_tokens_set}")

two_enc = base_tok.encode(sentence, allowed_special="all")
print(two_enc[:48])
print(base_tok.decode(two_enc[:48]))

for token_id in two_enc[:10]:
    print(f"{token_id} -> {base_tok.decode([token_id])}")

tiktoken.list_encoding_names()=['gpt2', 'r50k_base', 'p50k_base', 'p50k_edit', 'cl100k_base', 'o200k_base', 'o200k_harmony']
base_tok._pat_str="'(?:[sdmt]|ll|ve|re)| ?\\p{L}++| ?\\p{N}++| ?[^\\s\\p{L}\\p{N}]++|\\s++$|\\s+(?!\\S)|\\s"
base_tok.n_vocab=50257
mergable_rank=[{b'ters': 1010}, {b' take': 1011}, {b' Cl': 1012}, {b' conf': 1013}, {b'way': 1014}, {b'ave': 1015}, {b' going': 1016}, {b' sl': 1017}, {b'ug': 1018}, {b' Americ': 1019}]
special_tok={'<|endoftext|>'}
[2061, 2540, 287, 12122, 355, 257, 3210, 329, 2095, 21004, 468, 7334, 656, 257, 3665, 15320, 286, 1280, 2723, 5423, 11, 2438, 11, 4899, 11, 12782, 11, 290, 3186, 326, 4155, 3298, 3303, 1104, 11, 48817, 1799, 11, 290, 581, 2403, 1387, 1973, 13188, 286, 4410, 13, 198]
What began in 1988 as a standard for character encoding has grown into a powerful portfolio of open source standards, code, tools, libraries, and products that ensure global language support, interoperability, and resiliency across billions of devices.

2061 -

In [3]:
# Define custom tokens and their token IDs
custom_tokens = set([ENDOFPROMPT, FIM_MIDDLE, FIM_PREFIX, FIM_SUFFIX])
custom_token_ids = {
    token: base_tok.n_vocab + i for i, token in enumerate(custom_tokens)
}
print(f"{custom_token_ids=}")

# Create a new Encoding object with extended tokens
extended_tokenizer = tiktoken.Encoding(
    name="gpt2_custom",
    pat_str=base_tok._pat_str,
    mergeable_ranks=base_tok._mergeable_ranks,
    special_tokens={**base_tok._special_tokens, **custom_token_ids},
)

print(f"{extended_tokenizer.decode_bytes(tokens=[50257])=}")

custom_token_ids={'<|fim_suffix|>': 50257, '<|fim_prefix|>': 50258, '<|endofprompt|>': 50259, '<|fim_middle|>': 50260}
extended_tokenizer.decode_bytes(tokens=[50257])=b'<|fim_suffix|>'


### BPE

In [4]:
from tokenizers.decoders import BPEDecoder, WordPiece as WordPieceDecoder
from tokenizers.models import BPE
from tokenizers.normalizers import NFC, Sequence, Lowercase, StripAccents
from tokenizers.pre_tokenizers import (
    Digits,
    Punctuation,
    Whitespace,
    Split,
)
from tokenizers.processors import TemplateProcessing
from tokenizers import Tokenizer, pre_tokenizers
from tokenizers.trainers import BpeTrainer

from typing import Generator

In [5]:
file_path: str = r"/home/muthu/DATA/austen-emma.txt"


def get_file(path: str) -> str:
    with open(file_path, "r") as f:
        txt: str = f.read()
    return txt


def file_iterator(path: str) -> Generator:
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            yield line.strip()

In [6]:
special_tokens: list[str] = ["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"]

# Instantiate BPE (Byte-Pair Encoding)
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))

# a unicode normalizer, lowercasing and , replacing accents in order  :
# * Sequence : It composes multiple PreTokenizer that will be run in the given order
tokenizer.normalizer = Sequence([NFC(), Lowercase(), StripAccents()])  # ,NFKD(),NFKC(),

# Splits on word boundaries using the regular expression
# Whitespace: \w+|[^\w\s]+
tokenizer.pre_tokenizer = pre_tokenizers.Sequence(
    [
        Digits(),
        Punctuation(behavior="merged_with_previous"),
        Split(
            pattern=r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+""",
            behavior="isolated",
        ),
    ]
)

tokenizer.decoder = BPEDecoder()
tokenizer.post_processor = TemplateProcessing(
    single="[CLS] $A [SEP]",
    special_tokens=[("[CLS]", 1), ("[SEP]", 2)],  # IN spl_tokens list, we keep order
)

# Trainer
trainer = BpeTrainer(vocab_size=5000, special_tokens=special_tokens, show_progress=True)

In [7]:
%%time

tokenizer.train(files=[file_path], trainer=trainer)
print(f"Total number of Vocab::{tokenizer.get_vocab_size()}")




Total number of Vocab::5000
CPU times: user 3.32 s, sys: 2.18 s, total: 5.5 s
Wall time: 730 ms


In [8]:
sen_enc = tokenizer.encode(sentence)
print(f"Output: {format(sen_enc.ids)}")
print([tokenizer.id_to_token(i) for i in tokenizer.encode(sentence).ids])

Output: [1, 395, 1841, 1172, 17, 0, 23, 23, 225, 4156, 230, 839, 108, 1794, 4683, 447, 677, 112, 459, 534, 274, 164, 2701, 2274, 411, 176, 988, 36, 45, 159, 67, 143, 1036, 79, 213, 210, 2629, 441, 182, 119, 677, 120, 348, 1558, 182, 6, 159, 676, 84, 39, 321, 96, 3472, 631, 178, 597, 298, 2995, 388, 126, 245, 2747, 2179, 809, 74, 156, 2313, 45, 234, 166, 275, 1404, 96, 197, 2047, 447, 65, 191, 4922, 63, 32, 163, 287, 697, 133, 280, 240, 624, 5, 5, 3143, 497, 395, 461, 823, 108, 2825, 405, 1094, 79, 33, 77, 69, 63, 220, 951, 2545, 2066, 677, 57, 205, 628, 90, 400, 13, 101, 2427, 14, 1862, 78, 137, 1036, 79, 4093, 120, 6, 1036, 1010, 230, 843, 536, 750, 675, 44, 226, 126, 2602, 1902, 79, 78, 425, 336, 76, 836, 240, 624, 902, 240, 677, 4936, 111, 99, 34, 34, 86, 3487, 118, 40, 775, 45, 234, 125, 2296, 55, 179, 111, 96, 121, 1094, 686, 1039, 18, 16, 356, 163, 287, 133, 280, 240, 187, 1342, 34, 53, 198, 154, 194, 1879, 2717, 53, 198, 331, 219, 1990, 55, 86, 50, 35, 93, 83, 42, 2883, 65, 3036

In [9]:
from tokenizers.models import WordPiece
from tokenizers.normalizers import NFC, Sequence, Lowercase, StripAccents
from tokenizers.pre_tokenizers import Digits
from tokenizers import Tokenizer
from tokenizers.trainers import WordPieceTrainer

In [10]:
trainer = WordPieceTrainer(
    vocab_size=1000, special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"]
)

special_tokens = ["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"]

tokenizer = Tokenizer(WordPiece())
tokenizer.normalizer = Sequence([NFC(), Lowercase(), StripAccents()])  # ,NFKD(),NFKC(),
tokenizer.pre_tokenizer = pre_tokenizers.Sequence(
    [Whitespace(), Digits(individual_digits=True)]
)
tokenizer.decoder = WordPieceDecoder()
tokenizer.train(files=[r"/home/muthu/DATA/austen-emma.txt"], trainer=trainer)

In [11]:
sen_enc = tokenizer.encode(sentence)
print(f"Output: {format(sen_enc.ids)}")
print([tokenizer.id_to_token(i) for i in tokenizer.encode(sentence).ids])

Output: [248, 523, 113, 119, 15, 0, 21, 21, 147, 29, 221, 307, 266, 148, 743, 644, 93, 245, 55, 172, 103, 433, 455, 564, 551, 29, 453, 622, 69, 413, 453, 69, 75, 65, 347, 58, 62, 108, 869, 168, 176, 143, 221, 307, 266, 60, 11, 416, 467, 11, 430, 66, 60, 11, 261, 70, 385, 153, 107, 11, 105, 459, 72, 57, 166, 60, 144, 245, 60, 292, 723, 62, 70, 173, 40, 841, 57, 387, 528, 249, 11, 580, 260, 93, 419, 326, 353, 11, 105, 425, 326, 58, 104, 953, 351, 556, 60, 60, 30, 177, 498, 108, 194, 80, 509, 60, 13, 985, 285, 248, 435, 665, 148, 455, 243, 100, 156, 864, 434, 61, 60, 99, 72, 145, 171, 98, 310, 230, 172, 59, 461, 249, 58, 505, 11, 98, 117, 61, 12, 459, 384, 75, 869, 168, 176, 143, 11, 869, 221, 307, 266, 60, 499, 148, 98, 580, 61, 222, 173, 58, 82, 222, 108, 168, 812, 71, 452, 105, 330, 69, 80, 509, 60, 13, 310, 230, 172, 59, 171, 191, 873, 72, 467, 72, 119, 297, 41, 64, 84, 109, 542, 93, 102, 103, 47, 76, 140, 59, 73, 105, 426, 100, 156, 283, 278, 16, 14, 30, 177, 141, 194, 80, 509, 60, 8